# Población municipal del DANE

Abrí `muestras/dane-poblacion-municipal-2018-2042.xlsx`, hoja `PobMunicipalxÁrea`. Quería la fila de Medellín en 2021 y comprobar que el área Total no se sume con la cabecera.


In [1]:
from pathlib import Path
import pandas as pd

ruta = Path("..") / "muestras" / "dane-poblacion-municipal-2018-2042.xlsx"
pd.ExcelFile(ruta).sheet_names


['Índice', 'PPED', 'PobMunicipalxÁrea']

In [2]:
vista = pd.read_excel(ruta, sheet_name="PobMunicipalxÁrea", header=None, nrows=12)
vista


,0,1,2,3,4,5,6
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PROYECCIONES DE POBLACIÓN Y ESTUDIOS DEMOGRÁFI...,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MUNICIPAL. PROYECCIONES DE POBLACIÓN POR ÁREA ...,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,PROYECCIONES DE POBLACIÓN POR ÁREA. AÑOS 2018 ...,NaN,NaN,NaN,NaN,NaN,NaN
7,DP,DPNOM,MPIO,DPMP,AÑO,ÁREA GEOGRÁFICA,TOTAL
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,05,Antioquia,05001,Medellín,2018,Cabecera Municipal,2382405


Las primeras filas son el título de la proyección. La fila de columnas es la 8 del Excel, el índice 7. Debajo hay una fila en blanco. Si dejo que pandas adivine el encabezado, no encuentra `MPIO`.


In [3]:
poblacion = pd.read_excel(ruta, sheet_name="PobMunicipalxÁrea", header=7, dtype={"MPIO": str, "DP": str})
poblacion = poblacion.dropna(subset=["MPIO"]).copy()
poblacion["MPIO"] = poblacion["MPIO"].str.replace(r"\.0$", "", regex=True).str.zfill(5)
poblacion["AÑO"] = poblacion["AÑO"].astype(int)
poblacion["TOTAL"] = pd.to_numeric(poblacion["TOTAL"])
poblacion.shape


(84225, 7)

In [4]:
poblacion["AÑO"].min(), poblacion["AÑO"].max(), poblacion["MPIO"].nunique()


(np.int64(2018), np.int64(2042), 1123)

In [5]:
poblacion["ÁREA GEOGRÁFICA"].value_counts()


ÁREA GEOGRÁFICA
Cabecera Municipal                   28075
Centros Poblados y Rural Disperso    28075
Total                                28075
Name: count, dtype: int64

In [6]:
medellin_2021 = poblacion[(poblacion["MPIO"] == "05001") & (poblacion["AÑO"] == 2021)]
medellin_2021[["DPMP", "ÁREA GEOGRÁFICA", "TOTAL"]]


,DPMP,ÁREA GEOGRÁFICA,TOTAL
10108,Medellín,Cabecera Municipal,2459249.0
10109,Medellín,Centros Poblados y Rural Disperso,48371.0
10110,Medellín,Total,2507620.0


In [7]:
divipola = pd.read_csv(Path("..") / "muestras" / "divipola-codigos-municipios-20261007.csv", dtype=str)
solo_en_poblacion = sorted(set(poblacion["MPIO"]) - set(divipola["Código Municipio"]))
poblacion.loc[poblacion["MPIO"].isin(solo_en_poblacion), ["DPNOM", "MPIO", "DPMP"]].drop_duplicates()


,DPNOM,MPIO,DPMP
3307,Guainía,94663,Mapiripana (ANM)


Después de quitar la fila en blanco me quedaron 84.225 filas, de 2018 a 2042. Cada fila es un municipio, un año y un área. Las áreas son tres: Cabecera Municipal, Centros Poblados y Rural Disperso, y Total.

Medellín en 2021, código `05001`: cabecera 2.459.249, resto 48.371, total 2.507.620. Sumé cabecera y resto y me dio el total. No voy a sumar las tres filas.

Leí `MPIO` como texto. Si lo dejo numérico, `05001` se vuelve `5001` y pierdo el cero.

Comparé los códigos con el DIVIPOLA. Uno no está en ese padrón: `94663`, Mapiripana, un área no municipalizada de Guainía. No toca a Medellín. Esta hoja es contexto del municipio. No es una lista de clientes.
